In [1]:
import os

In [2]:
%pwd

'/Users/srishtisingh/chest-cancer-classification-mlops/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/srishtisingh/chest-cancer-classification-mlops'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list 

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chest-CT-Scan-data")

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile 
import tensorflow as tf
import time

In [9]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs

            )
        else:
            train_datagenerator = valid_datagenerator

            self.train_generator = train_datagenerator.flow_from_directory(
                directory=self.config.training_data,
                subset="training",
                shuffle=True,
                **dataflow_kwargs
            )

    @staticmethod
    def save_model(path:Path, model:tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [10]:
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.model_trainer import Training

try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-01-25 10:06:15,876: INFO: common: yaml file: /Users/srishtisingh/chest-cancer-classification-mlops/config/config.yaml loaded successfully]
[2026-01-25 10:06:15,878: INFO: common: yaml file: /Users/srishtisingh/chest-cancer-classification-mlops/params.yaml loaded successfully]
[2026-01-25 10:06:15,878: INFO: common: created directory at: artifacts]
[2026-01-25 10:06:15,878: INFO: common: created directory at: artifacts/training]


2026-01-25 10:06:15.897491: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-25 10:06:15.897515: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-25 10:06:15.897522: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-25 10:06:15.897558: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-25 10:06:15.897574: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


[2026-01-25 10:06:16,300: WARNING: optimizer: At this time, the v2.11+ optimizer `tf.keras.optimizers.SGD` runs slowly on M1/M2 Macs, please use the legacy Keras optimizer instead, located at `tf.keras.optimizers.legacy.SGD`.]
[2026-01-25 10:06:16,302: WARNING: __init__: There is a known slowdown when using v2.11+ Keras optimizers on M1/M2 Macs. Falling back to the legacy Keras optimizer, i.e., `tf.keras.optimizers.legacy.SGD`.]
✓ Model loaded successfully from: artifacts/prepare_base_model/base_model_updated.h5
Found 68 images belonging to 2 classes.
✓ Data augmentation enabled
Found 275 images belonging to 2 classes.
✓ Training samples: 275
✓ Validation samples: 68

Starting Training
Epochs: 20
Steps per epoch: 17
Validation steps: 4

Epoch 1/20


2026-01-25 10:06:16.787562: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


17/17 [==============================] - ETA: 0s - loss: 13.8340 - accuracy: 0.5483

2026-01-25 10:06:19.655576: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


17/17 [==============================] - 4s 190ms/step - loss: 13.8340 - accuracy: 0.5483 - val_loss: 9.3170 - val_accuracy: 0.3906
Epoch 2/20
17/17 [==============================] - 3s 173ms/step - loss: 11.7263 - accuracy: 0.5753 - val_loss: 0.0967 - val_accuracy: 0.9375
Epoch 3/20
17/17 [==============================] - 3s 175ms/step - loss: 7.1406 - accuracy: 0.6564 - val_loss: 21.8248 - val_accuracy: 0.3906
Epoch 4/20
17/17 [==============================] - 3s 174ms/step - loss: 9.9760 - accuracy: 0.5985 - val_loss: 0.1702 - val_accuracy: 0.9688
Epoch 5/20
17/17 [==============================] - 3s 185ms/step - loss: 2.0770 - accuracy: 0.8147 - val_loss: 4.8248 - val_accuracy: 0.6719
Epoch 6/20
17/17 [==============================] - 3s 173ms/step - loss: 7.1517 - accuracy: 0.6564 - val_loss: 0.0125 - val_accuracy: 1.0000
Epoch 7/20
17/17 [==============================] - 3s 175ms/step - loss: 0.8082 - accuracy: 0.8996 - val_loss: 0.0216 - val_accuracy: 1.0000
Epoch 8/20
17/

/opt/anaconda3/envs/chestmlops/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
